In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `llama-access` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llama

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 14.9 MB/s eta 0:00:00


In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
import torch

In [ ]:
import json

with open("final_adhd_dataset-train2.json", "r") as f:
    data = json.load(f)

# Ensure 'output' is always a string
for entry in data:
    if isinstance(entry["output"], list):
        entry["output"] = "\n".join(entry["output"])  # or ", ".join(entry["output"])

# Save as JSONL
with open("train.jsonl", "w") as f:
    for entry in data:
        f.write(json.dumps(entry) + "\n")

print(f"✅ Cleaned and converted {len(data)} entries to JSONL")

✅ Cleaned and converted 5578 entries to JSONL


In [ ]:
dataset_path = "train.jsonl"  # change to your dataset path
dataset = load_dataset("json", data_files={"train": dataset_path, "test": dataset_path})

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5578
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5578
    })
})


In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"  # or LLaMA 3 2B
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # LLaMA models don't have pad_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
max_length = 512

def tokenize_fn(example):
    # Concatenate instruction + input + output in a "prompt -> completion" style for supervised fine-tuning
    prompt_text = f"Instruction: {example['instruction']}\nInput: {example['input']}\nOutput: {example['output']}"
    enc = tokenizer(prompt_text, truncation=True, max_length=max_length, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()  # For causal LM
    return enc

tokenized_datasets = dataset.map(tokenize_fn, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/5578 [00:00<?, ? examples/s]

Map:   0%|          | 0/5578 [00:00<?, ? examples/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16  # T4 works best with float16
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],  # common for LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
training_args = TrainingArguments(
    output_dir="./llama3-adhd-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipython-input-34459727.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: siddarthaa (playground-siddarthaa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,1.383500
100,0.355600
150,0.318400
200,0.300100
250,0.290400
300,0.289800
350,0.283200
400,0.280200
450,0.266000
500,0.276200


TrainOutput(global_step=2094, training_loss=0.2907877879247511, metrics={'train_runtime': 3109.8459, 'train_samples_per_second': 5.381, 'train_steps_per_second': 0.673, 'total_flos': 5.007017125085184e+16, 'train_loss': 0.2907877879247511, 'epoch': 3.0})

In [ ]:
model.save_pretrained("./llama3-adhd-lora")

In [19]:
from transformers import pipeline

In [20]:
pipe = pipeline(
    "text-generation",
    model="./llama3-adhd-lora",
    tokenizer=tokenizer,
    device=0
)

Device set to use cuda:0


In [21]:
prompt = "Instruction: Organize the messy input into a structured task list.\nInput: I should probably clean Mom, check report, finish schedule, oh email presentation..."
output = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])

Instruction: Organize the messy input into a structured task list.
Input: I should probably clean Mom, check report, finish schedule, oh email presentation... oh submit assignment, I forgot to buy groceries.
Output: 1. Email presentation (15 min). 2. Submit assignment (10 min). 3. Check report (20 min). 4. Finish schedule (30 min). 5. Buy groceries (45 min). 6. Clean for Mom (30 min).
